In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

import joblib

In [2]:
df = pd.read_csv(r"C:\Users\ELCOT\Desktop\Sasirekha-G40 AIML 7\data\SmartShop_Portfolio_Dataset.csv")

df.head()

,Interaction_ID,Customer_ID,Product_ID,Category,Brand,Age,Gender,Region,Membership,Season,...,Recommendation_Score,Purchase_Status,Product_Rating,Customer_Lifetime_Value,Churn_Risk,Campaign_Response,Device,Payment_Method,Stock_Level,Delivery_Days
0,1,100001,4582,Books,PrimeGear,56,Male,South,Silver,Winter,...,54,0,3.7,138.29,42,1,Mobile,Card,357,7
1,2,100002,8527,Fashion,FreshMart,23,Female,East,Bronze,Summer,...,53,0,4.0,176.18,42,0,Desktop,UPI,437,4
2,3,100003,3677,Toys,FreshMart,35,Male,East,Gold,Spring,...,91,1,4.7,4254.82,7,1,Desktop,UPI,108,10
3,4,100004,5304,Grocery,EcoLife,43,Female,South,Gold,Summer,...,83,1,3.9,3154.76,20,0,Mobile,Card,405,7
4,5,100005,6573,Fashion,BookNest,45,Male,West,Silver,Spring,...,90,1,5.0,3594.27,15,0,Tablet,Wallet,430,9


In [3]:
print(df.shape)

(15000, 30)


In [4]:
duplicates = df.duplicated().sum()

print("Duplicate Rows :", duplicates)

df = df.drop_duplicates()

print(df.shape)

Duplicate Rows : 0
(15000, 30)


In [5]:
df.isnull().sum()

Interaction_ID              0
Customer_ID                 0
Product_ID                  0
Category                    0
Brand                       0
Age                         0
Gender                      0
Region                      0
Membership                  0
Season                      0
Holiday                     0
Price                       0
Discount                    0
Views                       0
Time_On_Page                0
Wishlist                    0
Added_To_Cart               0
Previous_Purchases          0
Average_Order_Value         0
Days_Since_Last_Purchase    0
Recommendation_Score        0
Purchase_Status             0
Product_Rating              0
Customer_Lifetime_Value     0
Churn_Risk                  0
Campaign_Response           0
Device                      0
Payment_Method              0
Stock_Level                 0
Delivery_Days               0
dtype: int64

In [6]:
numerical_features = df.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

categorical_features = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features")
print(numerical_features)

print()

print("Categorical Features")
print(categorical_features)

Numerical Features
['Interaction_ID', 'Customer_ID', 'Product_ID', 'Age', 'Holiday', 'Price', 'Discount', 'Views', 'Time_On_Page', 'Wishlist', 'Added_To_Cart', 'Previous_Purchases', 'Average_Order_Value', 'Days_Since_Last_Purchase', 'Recommendation_Score', 'Purchase_Status', 'Product_Rating', 'Customer_Lifetime_Value', 'Churn_Risk', 'Campaign_Response', 'Stock_Level', 'Delivery_Days']

Categorical Features
['Category', 'Brand', 'Gender', 'Region', 'Membership', 'Season', 'Device', 'Payment_Method']


In [7]:
regression_target = "Product_Rating"

In [8]:
classification_target = "Purchase_Status"

In [9]:
numerical_features.remove(regression_target)
numerical_features.remove(classification_target)

In [10]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    )
])

In [11]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

In [13]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_pipeline,
            numerical_features
        ),

        (
            "cat",
            categorical_pipeline,
            categorical_features
        )

    ]

)

In [14]:
X_reg = df.drop(
    columns=["Product_Rating"]
)

y_reg = df["Product_Rating"]

In [15]:
X_cls = df.drop(
    columns=["Purchase_Status"]
)

y_cls = df["Purchase_Status"]

In [16]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(

    X_reg,

    y_reg,

    test_size=0.20,

    random_state=42

)

In [17]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(

    X_cls,

    y_cls,

    test_size=0.20,

    random_state=42,

    stratify=y_cls

)

In [18]:
X_train_reg_processed = preprocessor.fit_transform(
    X_train_reg
)

X_test_reg_processed = preprocessor.transform(
    X_test_reg
)

In [19]:
X_train_cls_processed = preprocessor.fit_transform(
    X_train_cls
)

X_test_cls_processed = preprocessor.transform(
    X_test_cls
)

In [20]:
print(X_train_reg_processed.shape)

print(X_test_reg_processed.shape)

print(X_train_cls_processed.shape)

print(X_test_cls_processed.shape)

(12000, 59)
(3000, 59)
(12000, 59)
(3000, 59)


In [21]:
joblib.dump(

    preprocessor,

    "../models/preprocessor.pkl"

)

['../models/preprocessor.pkl']

In [22]:
joblib.dump(

    X_train_reg_processed,

    "../models/X_train_reg.pkl"

)

joblib.dump(

    X_test_reg_processed,

    "../models/X_test_reg.pkl"

)

['../models/X_test_reg.pkl']

Data Preprocessing Summary
Removed duplicate records.
Identified numerical and categorical features.
Imputed missing numerical values using the median.
Imputed missing categorical values using the most frequent category.
Applied StandardScaler to numerical variables.
Applied One-Hot Encoding to categorical variables.
Split the data into training and testing sets.
Saved the preprocessing pipeline for reuse.